# LeetCode #269: Alien Dictionary

https://leetcode.com/problems/alien-dictionary/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Topological Sort (BFS/Kahn's) ★ | O(C) | O(U + E) |
| Topological Sort (DFS) | O(C) | O(U + E) |

## Understanding the Methods
### Topological Sort with BFS (Optimal)
Compare adjacent words to extract ordering constraints between characters. Build a directed graph and compute in-degrees. Use BFS (Kahn's algorithm) to produce a topological ordering. If the result includes all unique characters, the ordering is valid. C is total characters across all words, U is unique letters, E is edges.

### Topological Sort with DFS
Same graph construction, but use DFS with cycle detection (three-state coloring). Post-order reversal gives the topological order. Detects cycles when a gray node is revisited.

## Solutions

### C#

In [ ]:
public class Solution {
    public string AlienOrder(string[] words) {
        var adj = new Dictionary<char, HashSet<char>>();
        var inDeg = new Dictionary<char, int>();
        foreach (var w in words)
            foreach (var c in w) {
                if (!adj.ContainsKey(c)) adj[c] = new HashSet<char>();
                if (!inDeg.ContainsKey(c)) inDeg[c] = 0;
            }

        for (int i = 0; i < words.Length - 1; i++) {
            string w1 = words[i], w2 = words[i + 1];
            if (w1.Length > w2.Length && w1.StartsWith(w2)) return "";
            for (int j = 0; j < Math.Min(w1.Length, w2.Length); j++) {
                if (w1[j] != w2[j]) {
                    if (adj[w1[j]].Add(w2[j]))
                        inDeg[w2[j]]++;
                    break;
                }
            }
        }

        var queue = new Queue<char>();
        foreach (var kv in inDeg)
            if (kv.Value == 0) queue.Enqueue(kv.Key);

        var sb = new System.Text.StringBuilder();
        while (queue.Count > 0) {
            char c = queue.Dequeue();
            sb.Append(c);
            foreach (var nei in adj[c]) {
                inDeg[nei]--;
                if (inDeg[nei] == 0) queue.Enqueue(nei);
            }
        }
        return sb.Length == inDeg.Count ? sb.ToString() : "";
    }
}

### Python

In [ ]:
from collections import defaultdict, deque

class Solution:
    def alienOrder(self, words: list[str]) -> str:
        adj = defaultdict(set)
        in_deg = {c: 0 for w in words for c in w}

        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i + 1]
            if len(w1) > len(w2) and w1.startswith(w2):
                return ""
            for a, b in zip(w1, w2):
                if a != b:
                    if b not in adj[a]:
                        adj[a].add(b)
                        in_deg[b] += 1
                    break

        queue = deque(c for c in in_deg if in_deg[c] == 0)
        result = []
        while queue:
            c = queue.popleft()
            result.append(c)
            for nei in adj[c]:
                in_deg[nei] -= 1
                if in_deg[nei] == 0:
                    queue.append(nei)
        return "".join(result) if len(result) == len(in_deg) else ""

### Go

In [ ]:
func alienOrder(words []string) string {
    adj := map[byte]map[byte]bool{}
    inDeg := map[byte]int{}
    for _, w := range words {
        for i := 0; i < len(w); i++ {
            if _, ok := adj[w[i]]; !ok {
                adj[w[i]] = map[byte]bool{}
            }
            inDeg[w[i]] += 0
        }
    }

    for i := 0; i < len(words)-1; i++ {
        w1, w2 := words[i], words[i+1]
        minLen := len(w1)
        if len(w2) < minLen { minLen = len(w2) }
        if len(w1) > len(w2) && w1[:len(w2)] == w2 {
            return ""
        }
        for j := 0; j < minLen; j++ {
            if w1[j] != w2[j] {
                if !adj[w1[j]][w2[j]] {
                    adj[w1[j]][w2[j]] = true
                    inDeg[w2[j]]++
                }
                break
            }
        }
    }

    queue := []byte{}
    for c := range inDeg {
        if inDeg[c] == 0 {
            queue = append(queue, c)
        }
    }

    result := []byte{}
    for len(queue) > 0 {
        c := queue[0]
        queue = queue[1:]
        result = append(result, c)
        for nei := range adj[c] {
            inDeg[nei]--
            if inDeg[nei] == 0 {
                queue = append(queue, nei)
            }
        }
    }
    if len(result) != len(inDeg) {
        return ""
    }
    return string(result)
}

### Rust

In [ ]:
use std::collections::{HashMap, HashSet, VecDeque};

impl Solution {
    pub fn alien_order(words: Vec<String>) -> String {
        let mut adj: HashMap<u8, HashSet<u8>> = HashMap::new();
        let mut in_deg: HashMap<u8, i32> = HashMap::new();
        for w in &words {
            for &b in w.as_bytes() {
                adj.entry(b).or_default();
                in_deg.entry(b).or_insert(0);
            }
        }

        for i in 0..words.len() - 1 {
            let (w1, w2) = (words[i].as_bytes(), words[i + 1].as_bytes());
            if w1.len() > w2.len() && w1.starts_with(w2) {
                return String::new();
            }
            for j in 0..w1.len().min(w2.len()) {
                if w1[j] != w2[j] {
                    if adj.get_mut(&w1[j]).unwrap().insert(w2[j]) {
                        *in_deg.get_mut(&w2[j]).unwrap() += 1;
                    }
                    break;
                }
            }
        }

        let mut queue: VecDeque<u8> = in_deg.iter()
            .filter(|(_, &v)| v == 0).map(|(&k, _)| k).collect();
        let mut result = Vec::new();
        while let Some(c) = queue.pop_front() {
            result.push(c);
            if let Some(neighbors) = adj.get(&c) {
                for &nei in neighbors {
                    *in_deg.get_mut(&nei).unwrap() -= 1;
                    if in_deg[&nei] == 0 {
                        queue.push_back(nei);
                    }
                }
            }
        }
        if result.len() != in_deg.len() { String::new() }
        else { String::from_utf8(result).unwrap() }
    }
}

## Example Scenarios

### 1. Standard Order
**Input:** `words = ["wrt","wrf","er","ett","rftt"]`  
Derived edges: t->f, w->e, r->t, e->r. **Output:** `"wertf"`

### 2. Simple Two-Letter
**Input:** `words = ["z","x"]`  
z comes before x. **Output:** `"zx"`

### 3. Invalid Order (Cycle)
**Input:** `words = ["z","x","z"]`  
z < x and x < z is contradictory. **Output:** `""`

### 4. Invalid Prefix
**Input:** `words = ["abc","ab"]`  
Longer word before its prefix is invalid. **Output:** `""`

### 5. Single Word
**Input:** `words = ["abc"]`  
No ordering constraints, any permutation of a,b,c is valid.

![image](attachment:image.png)